In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import lingam
import networkx as nx
import matplotlib.pyplot as plt

#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set NIJ root
nij_root = project_root / "data" / "processed"

#path to put results
output_dir = project_root/"results"/"NIJ"/"graphs_LiM"
output_dir.mkdir(parents=True,exist_ok=True)

In [2]:
import time
"""LiM expects a numeric matrix with continuous and discrete columns as well as
an array per variable indicating 0 for discrete and 1 for continuous.
Similar to what we did for DAGBagM, we write a helper function to infer the datatype."""

def infer_type(df):
    """
    Infers a 'flag array' for a given dataframe.
    0 corresponds to discrete variables, 1 to continuous variables
    """

    flags=[]
    for col in df.columns:
        x = df[col].dropna()
        #classify numeric columns
        if np.issubdtype(x.dtype, np.number):
            vals = x.unique()
            if len(vals) <= 1:
                flags.append(0) #variable is discrete
            else:
                flags.append(1) # variable is continuous
    return np.asarray(flags, dtype=int)

def run_lim(df, seed=1):
    """
    Clean dataframe and generate the numeric matrix as well as flag array.
    Run LiM on the data.
    """

    df_clean=df.dropna().copy()
    flags = infer_type(df_clean)
    flags_2d = flags.reshape(1,-1)
    cols = list(df_clean.columns)

    X = df_clean.to_numpy(dtype=float)

    start = time.time()
    model = lingam.LiM()

    model.fit(X, flags_2d, only_global=True)
    #LiM creates an adjacency matrix
    M = model.adjacency_matrix_
    #coerce adjacency matrix into a matrix of 0's and 1's
    adj = (np.abs(M) > 0).astype(int)
    end=time.time()
    
    #print(f"LiM took {(end - start)/60:.2f} minutes")
    return adj, cols

In [3]:
def encode_mixed_df(df):
    df_enc = df.copy()
    for col in df_enc.columns:
        #use categorical code for non numeric
        if not np.issubdtype(df_enc[col].dtype, np.number):
            df_enc[col] = df_enc[col].astype("category").cat.codes
    return df_enc

In [5]:
csv_path = output_dir/ "NIJ_graph_LiM_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
A = adj_df.to_numpy()

import itertools
import random
import time

def incompatibility_score(W_full, k=5, n_subsets=50, seed=0):
    #sample random k-node subsets, run the algorithm again, and compare compatibility with learned DAG
    rng = random.Random(seed)
    W_full = (np.asarray(W_full) != 0).astype(int)
    d = W_full.shape[0]

    if k > d:
        raise ValueError("Subset size k cannot exceed number of variables")

    #computes SHD between two adjacency matrices
    def shd(A, B):
        A = (A != 0).astype(int)
        B = (B != 0).astype(int)
        return np.sum(A != B)

    shd_vals = []

    for _ in range(n_subsets):
        #randomly sample a k-variable subset of all variables
        idx = sorted(rng.sample(range(d), k))

        # 1)restrict the full graph to this subset
        W_restricted = W_full[np.ix_(idx, idx)]

        # 2)run cd algorithm on subset of variables
        csv_path= nij_root/"NIJ_lean_compact_onehot.csv"
        df = pd.read_csv(csv_path)
        df_subset = df.iloc[:, idx]
        df_subset_enc = encode_mixed_df(df_subset)
        adj, _ = run_lim(df_subset_enc, seed=1)
 
        # 3) compute SHD between the restricted full DAG and the subset DAG
        shd_vals.append(shd(W_restricted, adj))

    # incompatibility score ~= average SHD across subsets
    return np.mean(shd_vals)

score = incompatibility_score(A, k=5, n_subsets=50, seed=42)
print("Approx. incompatibility score:", score)

W_est (without the 2nd phase) is: 
 [[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
LiM took 0.02 minutes
W_est (without the 2nd phase) is: 
 [[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
LiM took 0.04 minutes
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.13348858 0.         0.         0.        ]]
LiM took 0.01 minutes


/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.         44.40926686  0.        ]]
LiM took 0.03 minutes
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.15520118]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.20470497 0.         0.        ]]
LiM took 0.01 minutes
W_est (without the 2nd phase) is: 
 [[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
LiM took 0.03 minutes
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.         0.        ]
 [0.         0.         0.        

/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          5.96324051]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.         26.77121028]
 [ 0.          0.          0.          0.          0.        ]]
LiM took 0.04 minutes
W_est (without the 2nd phase) is: 
 [[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
LiM took 0.00 minutes
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         5.55501712]
 [0.         0.         0.         0.         2.7538005 ]
 [0.         0.         0.         0.         0.        ]]
LiM took 0.03 minutes


...

W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.         27.76846917  0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]]
LiM took 0.04 minutes
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         2.98452102]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         4.74784153]
 [0.         0.         0.         0.         0.        ]]
LiM took 0.02 minutes


/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.          0.          1.5199908 ]
 [ 0.          0.          0.          0.          8.63509837]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.         47.79444309]
 [ 0.          0.          0.          0.          0.        ]]
LiM took 0.04 minutes
W_est (without the 2nd phase) is: 
 [[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
LiM took 0.04 minutes
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.33479628 0.         0.        ]]
LiM took 0.01 minutes
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.         0.        ]
 [0.         0.         0.        

/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.         27.76846925]
 [ 0.          0.          0.          0.          0.        ]]
LiM took 0.04 minutes
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.11797489 0.         0.        ]]
LiM took 0.01 minutes
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.12862447 0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.     

In [6]:
def goodness(incompat_score, k):
    poss_edges = k*(k-1)
    frac = incompat_score/poss_edges
    print("% of edges disagreeing on avg: " + str(frac*100))

goodness(score, 5)

% of edges disagreeing on avg: 4.7


In [4]:
csv_path = output_dir/ "NIJ_graph_LiM_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
A = adj_df.to_numpy()

import itertools
import random

def incompatibility_score(W_full, k=5, n_subsets=50, seed=0):
    #sample random k-node subsets, run the algorithm again, and compare compatibility with learned DAG
    rng = random.Random(seed)
    W_full = (np.asarray(W_full) != 0).astype(int)
    d = W_full.shape[0]

    if k > d:
        raise ValueError("Subset size k cannot exceed number of variables")

    #computes SHD between two adjacency matrices
    def shd(A, B):
        A = (A != 0).astype(int)
        B = (B != 0).astype(int)
        return np.sum(A != B)

    shd_vals = []

    for _ in range(n_subsets):
        #randomly sample a k-variable subset of all variables
        idx = sorted(rng.sample(range(d), k))

        # 1)restrict the full graph to this subset
        W_restricted = W_full[np.ix_(idx, idx)]

        # 2)run cd algorithm on subset of variables
        csv_path= nij_root/"NIJ_lean_compact_onehot.csv"
        df = pd.read_csv(csv_path)
        df_subset = df.iloc[:, idx]
        df_subset_enc = encode_mixed_df(df_subset)
        adj, _ = run_lim(df_subset_enc, seed=1)
 
        # 3) compute SHD between the restricted full DAG and the subset DAG
        shd_vals.append(shd(W_restricted, adj))

    # incompatibility score ~= average SHD across subsets
    return np.mean(shd_vals)

def goodness(incompat_score, k):
    poss_edges = k*(k-1)
    frac = incompat_score/poss_edges
    print("% of edges disagreeing on avg: " + str(frac*100))
    return(frac*100)

In [5]:
csv_path = output_dir/ "NIJ_graph_LiM_adj.csv"
adj_df = pd.read_csv(csv_path, index_col=0)
node_labels = list(adj_df.index)
A = adj_df.to_numpy()

seeds = [42,7,12]
incompat_scores = []
disagree = []
for seed in seeds:
    score = incompatibility_score(A, k = 5, n_subsets=50, seed=seed)
    incompat_scores.append(score)
    disagree.append(goodness(score, 5))

print("standard dev of incompatability score: " + str(np.std(incompat_scores)))
print("standard dev of disagreement percentage: " + str(np.std(disagree)))

W_est (without the 2nd phase) is: 
 [[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
W_est (without the 2nd phase) is: 
 [[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
W_est (without the 2nd phase) is: 
 [[0.        0.        0.        0.        0.       ]
 [0.        0.        0.        0.        0.       ]
 [0.        0.        0.        0.        0.       ]
 [0.        0.        0.        0.        0.       ]
 [0.        0.1334886 0.        0.        0.       ]]


/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.         44.40926684  0.        ]]
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.15520118]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.20470497 0.         0.        ]]
W_est (without the 2nd phase) is: 
 [[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.     

/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          5.96323874]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.         26.77121082]
 [ 0.          0.          0.          0.          0.        ]]
W_est (without the 2nd phase) is: 
 [[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         5.54602975]
 [0.         0.         0.         0.         2.76858532]
 [0.         0.         0.         0.         0.        ]]


...

W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.         19.02508093  0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          8.732047    0.        ]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.         28.13427112  0.        ]]


/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.         35.36390598]
 [ 0.          0.          0.          0.          0.        ]
 [ 0.          0.3923438   0.          0.         24.27901792]
 [ 0.          0.          0.          0.          0.        ]]


/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         5.89410842]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         7.96912514 0.         0.        ]]
% of edges disagreeing on avg: 7.199999999999999
standard dev of incompatability score: 0.20417857108151405
standard dev of disagreement percentage: 1.02089285540757


In [6]:
print("standard dev of incompatability score: " + str(np.std(incompat_scores)))
print("standard dev of disagreement percentage: " + str(np.std(disagree)))

standard dev of incompatability score: 0.20417857108151405
standard dev of disagreement percentage: 1.02089285540757


In [7]:
incompat_scores

[0.94, 1.2, 1.44]

In [8]:
disagree

[4.7, 6.0, 7.199999999999999]